# pep_data Top 克隆提取

这个 notebook 用于：

- 读取当前工作区下 `./pep_data` 目录中的 pep 数据
- 按链目录循环处理，例如 `./pep_data/TRA/`、`./pep_data/TRB/`
- 对每个输入文件按 `copy` 降序提取 Top N 克隆
- 输出到 `./top_clones/<链>/` 目录下

输出字段统一为：`index`, `Chain`, `CDR3(pep)`, `joinedSeq`, `V`, `D`, `J`, `C`, `copy`


In [ ]:
from pathlib import Path

# 可修改配置
INPUT_ROOTS = [
    "./pep_data"
]

TOP_N = 10
OUTPUT_FOLDER_NAME = "top_clones"

if not INPUT_ROOTS:
    print("请先在 INPUT_ROOTS 中填写一个或多个输入路径。")
else:
    print("当前输入路径：")
    for root in INPUT_ROOTS:
        print(f"- {root}")
        print(f"  输出目录: {Path(root).parent / OUTPUT_FOLDER_NAME}")
    print(f"Top N = {TOP_N}")


CHAIN_LABEL_MAP = {
    "TRA": "Alpha",
    "TRB": "Beta",
    "TRG": "Gamma",
    "TRD": "Delta",
    "IGH": "Heavy",
    "IGK": "Kappa",
    "IGL": "Lambda",
}


In [ ]:
import re
from typing import Dict, List, Optional, Tuple

import pandas as pd

OUTPUT_COLUMNS = [
    "index",
    "Chain",
    "CDR3(pep)",
    "joinedSeq",
    "V",
    "D",
    "J",
    "C",
    "copy",
]

INPUT_COLUMNS = [
    "CDR3(pep)",
    "joinedSeq",
    "V",
    "D",
    "J",
    "C",
    "copy",
]

CHAIN_NAME_ALIASES = {
    "TRA": "TRA",
    "TRB": "TRB",
    "TRG": "TRG",
    "TRD": "TRD",
    "IGH": "IGH",
    "IGK": "IGK",
    "IGL": "IGL",
    "ALPHA": "TRA",
    "BETA": "TRB",
    "GAMMA": "TRG",
    "DELTA": "TRD",
    "HEAVY": "IGH",
    "KAPPA": "IGK",
    "LAMBDA": "IGL",
}

FILE_PATTERN = re.compile(
    r"^(?P<sample>.+?)(?:__|_|-)?(?P<chain>TRA|TRB|TRG|TRD|IGH|IGK|IGL)?$",
    re.IGNORECASE,
)


def normalize_chain_name(raw_name: str) -> str:
    normalized = re.sub(r"[^A-Za-z]", "", raw_name).upper()
    return CHAIN_NAME_ALIASES.get(normalized, raw_name.upper())


def remove_known_suffixes(file_name: str) -> str:
    lowered = file_name.lower()
    if lowered.endswith(".csv.gz"):
        return file_name[:-7]
    if lowered.endswith(".csv"):
        return file_name[:-4]
    return Path(file_name).stem


def parse_sample_name(file_path: Path, chain: str) -> str:
    base_name = remove_known_suffixes(file_path.name)
    match = FILE_PATTERN.match(base_name)
    if not match:
        return base_name

    sample = match.group("sample") or base_name
    parsed_chain = match.group("chain")
    if parsed_chain and normalize_chain_name(parsed_chain) == chain:
        return sample.rstrip("_-")
    return base_name


def discover_chain_directories(root: Path) -> List[Tuple[str, Path]]:
    chain_directories = []
    for entry in sorted(root.iterdir()):
        if not entry.is_dir():
            continue
        chain_directories.append((normalize_chain_name(entry.name), entry))
    return chain_directories


def iter_supported_files(root: Path):
    for file_path in sorted(root.rglob("*")):
        if not file_path.is_file():
            continue
        lowered = file_path.name.lower()
        if lowered.endswith(".csv") or lowered.endswith(".csv.gz"):
            yield file_path


def sanitize_sheet_frame(df: pd.DataFrame, chain: str, top_n: int) -> pd.DataFrame:
    missing_columns = [column for column in INPUT_COLUMNS if column not in df.columns]
    if missing_columns:
        raise ValueError(
            f"缺少必要字段: {missing_columns}; 实际字段为: {list(df.columns)}"
        )

    output = pd.DataFrame(index=df.index)
    output["Chain"] = CHAIN_LABEL_MAP.get(chain, chain)
    output["CDR3(pep)"] = df["CDR3(pep)"].fillna("").astype(str).str.strip()

    for field in ["joinedSeq", "V", "D", "J", "C"]:
        output[field] = df[field].fillna("").astype(str).str.strip()

    copy_series = (
        df["copy"].fillna("").astype(str).str.replace(",", "", regex=False).str.strip()
    )
    output["copy"] = pd.to_numeric(copy_series, errors="coerce")
    output = output.dropna(subset=["copy"])
    output = output[output["CDR3(pep)"] != ""]
    output = output.sort_values(
        by=["copy", "CDR3(pep)"], ascending=[False, True], kind="mergesort"
    )
    output = output.head(top_n).reset_index(drop=True)
    output.insert(0, "index", range(1, len(output) + 1))
    output = output[OUTPUT_COLUMNS]
    return output


def load_chain_top_table(file_path: Path, chain: str, top_n: int) -> pd.DataFrame:
    df = pd.read_csv(file_path, compression="infer")
    return sanitize_sheet_frame(df, chain=chain, top_n=top_n)


def make_output_file(
    output_root: Path,
    chain: str,
    chain_dir: Path,
    source_file: Path,
    sample: str,
    top_n: int,
) -> Path:
    relative_parent = source_file.relative_to(chain_dir).parent
    output_dir = output_root / chain / relative_parent
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir / f"{sample}_top{top_n}.csv"


In [ ]:
def process_root(root_path: str, top_n: int = TOP_N) -> pd.DataFrame:
    root = Path(root_path)
    if not root.exists():
        raise FileNotFoundError(f"输入路径不存在: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"输入路径不是目录: {root}")

    output_root = root.parent / OUTPUT_FOLDER_NAME
    chain_directories = discover_chain_directories(root)
    if not chain_directories:
        print(f"[跳过] 未在 {root} 下找到按链分类的子目录")
        return pd.DataFrame(
            columns=["root", "chain_dir", "chain", "sample", "file"]
        )

    output_records = []

    for chain, chain_dir in chain_directories:
        print(f"处理链目录: {chain_dir}")
        files = list(iter_supported_files(chain_dir))
        if not files:
            print("  - 当前链目录下未发现 csv/csv.gz 文件")
            continue

        for file_path in files:
            sample = parse_sample_name(file_path, chain)

            try:
                table = load_chain_top_table(file_path, chain=chain, top_n=top_n)
            except Exception as exc:
                print(f"  - {file_path.name} 读取失败: {exc}")
                continue

            output_file = make_output_file(
                output_root=output_root,
                chain=chain,
                chain_dir=chain_dir,
                source_file=file_path,
                sample=sample,
                top_n=top_n,
            )
            table.to_csv(output_file, index=False, encoding="utf-8-sig")
            output_records.append(
                {
                    "root": str(root),
                    "chain_dir": str(chain_dir),
                    "chain": chain,
                    "sample": sample,
                    "file": str(output_file),
                }
            )

    return pd.DataFrame(output_records)


In [ ]:
all_outputs = []

for root_path in INPUT_ROOTS:
    result = process_root(root_path, top_n=TOP_N)
    if not result.empty:
        all_outputs.append(result)

if all_outputs:
    summary_df = pd.concat(all_outputs, ignore_index=True)
    display(summary_df)
    summary_path = Path(INPUT_ROOTS[0]).parent / OUTPUT_FOLDER_NAME / "summary.csv"
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
    print(f"共生成 {len(summary_df)} 个输出文件")
    print(f"汇总文件: {summary_path}")
else:
    summary_df = pd.DataFrame(columns=["root", "chain_dir", "chain", "sample", "file"])
    print("没有生成任何输出文件，请检查 pep_data 目录结构和字段名。")


## 使用说明

1. 确保输入目录为当前工作区下的 `./pep_data`。
2. `pep_data` 下按链放置数据，例如：`./pep_data/TRA/`、`./pep_data/TRB/`。
3. 按需修改 `TOP_N`。
4. 运行全部单元格。
5. 输出目录会生成为：`./top_clones/`。
6. 每条链的结果会写入对应子目录：`./top_clones/<链>/`。
7. 如果链目录下还有子目录结构，会在输出目录中保留同样的相对层级。
